# 10 — Generics & Variance

Up to now you've used types like `List[Int]`, `Option[String]`, and `Either[Error, User]` without dwelling on what the brackets actually mean. This notebook opens that box. **Generics** let you write code once that works for many element types. **Variance** then governs the subtype relationships between those parameterised types — when `Option[Cat]` should also count as an `Option[Animal]`, and when it shouldn't.

Two questions drive everything here:

1. How do I write a function or class that's parameterised over an element type `A`?
2. Given that I have such a type, how does `Box[Cat]` relate to `Box[Animal]`?

The first question is mostly mechanical. The second is where the interesting design decisions live — and where Scala gives you finer control than most languages.

## Why generics exist

Imagine you want a function that returns its argument unchanged. Without generics you'd write it once per type — `identityInt`, `identityString`, `identityUser`, on and on. Or you'd accept `Any` and lose all type information at the return site. Neither is acceptable.

Generics let you write the function *once*, with a placeholder `A` standing in for the actual type. The compiler then specialises that placeholder at every call site, preserving full type information without duplicating code.

## Generic methods

A method becomes generic by declaring one or more type parameters in square brackets right after the method name. Inside the body those parameters behave as ordinary types — you just don't know which concrete type they'll be at definition time.

In [ ]:
def identity[A](x: A): A = x

identity[Int](42)        // 42 — A is Int
identity("scala")        // "scala" — A inferred as String
identity(List(1, 2, 3))  // List(1, 2, 3) — A inferred as List[Int]

You almost never write the type parameter at the call site — Scala infers it from the argument. The explicit form `identity[Int](42)` is there as a fallback for when inference can't figure it out, usually because the argument is ambiguous or you want to widen the type on purpose.

Multiple type parameters work the same way. Each is independent, and each is inferred from whichever argument constrains it.

In [ ]:
def pair[A, B](a: A, b: B): (A, B) = (a, b)

pair(1, "one")           // (Int, String) inferred
pair(true, 3.14)         // (Boolean, Double) inferred
pair[Long, Long](1, 2)   // forced both sides to Long

## Generic classes

The same square-bracket syntax works on classes. A `Box[A]` is a class with one type parameter — every instance is a box of *some* specific `A`, but the class definition doesn't care which.

In [ ]:
case class Box[A](value: A):
  def map[B](f: A => B): Box[B] = Box(f(value))

val b1: Box[Int]    = Box(42)
val b2: Box[String] = b1.map(_.toString)   // Box("42")

Two things to notice. First, the *class* takes a type parameter `A`, and the *method* `map` introduces its own type parameter `B`. Class type parameters are fixed once the instance is created; method type parameters are fresh at every call. Second, the `map` method is the same shape you saw on `Option`, `Try`, and `Either` in notebook 09 — once you have generics, this kind of container with a `map` is a pattern you'll keep meeting.

## Type bounds — constraining what `A` can be

Sometimes a generic type parameter shouldn't accept *any* type. You need a method that adds two values, or compares them, or calls some method that only certain types have. Type bounds let you require that `A` is a subtype (or supertype) of something specific.

- **Upper bound** — `A <: T` means `A` must be `T` or a subtype of `T`.
- **Lower bound** — `A >: T` means `A` must be `T` or a supertype of `T`.

### Upper bounds

Upper bounds are the common case. They say: *I only accept `A`s that have at least these capabilities.*

In [ ]:
trait Named:
  def name: String

case class Person(name: String) extends Named
case class City(name: String)   extends Named

def greet[A <: Named](thing: A): String = s"hello, ${thing.name}"

greet(Person("alice"))   // "hello, alice"
greet(City("madrid"))    // "hello, madrid"
// greet(42)             // won't compile — Int isn't Named

Inside the body, `thing: A` is known to be at least a `Named`, so calling `thing.name` is allowed. The compiler still tracks the more specific `A` outside, so the return type carries that precision when needed.

### Lower bounds

Lower bounds are rarer in everyday code, but they're essential for one specific pattern: adding elements to a covariant collection. We'll come back to them when we get to variance below — the canonical example is `List#::` (cons), which uses a lower bound to widen the element type as you prepend.

Mental shorthand: an upper bound means *I'm restricted from above* — `A` is at most that type. A lower bound means *I'm restricted from below* — `A` is at least that type.

## Variance — the relationship between `Box[Cat]` and `Box[Animal]`

Here's the central question. Given a hierarchy `class Animal`, `class Cat extends Animal`, and a parameterised type `Box[A]`, three answers are possible for how `Box[Cat]` and `Box[Animal]` relate:

```
  invariant       Box[Cat] is NOT a Box[Animal] (and vice versa)
  covariant       Box[Cat] IS a Box[Animal]              (Box "follows" A)
  contravariant   Box[Animal] IS a Box[Cat]              (reversed!)
```

The default in Scala is invariant. Covariance is declared with a `+` on the type parameter; contravariance with a `-`. So `class Box[A]` is invariant, `class Box[+A]` is covariant, `class Box[-A]` is contravariant. Choosing wisely is what we're about to learn.

### Why invariance is the default

Imagine `Box[A]` were *automatically* covariant — that is, `Box[Cat]` could be used wherever `Box[Animal]` is expected. For a mutable box, this breaks type safety in obvious ways:

In [ ]:
// Pseudocode of a scenario the compiler must reject
//
// class MBox[A](var value: A)            // mutable, NOT covariant
// val catBox: MBox[Cat]    = MBox(Cat())
// val anyBox: MBox[Animal] = catBox      // would be allowed if covariant
// anyBox.value = Dog()                   // legal — Dog IS an Animal
// val cat: Cat = catBox.value            // crash — actually a Dog!

// Mutable cells must stay invariant to keep this scenario impossible.

This is the Liskov-substitution-meets-mutation problem. As soon as you can both *write* and *read* at the parameter type, allowing subtype substitution lets a writer using the wide view break a reader using the narrow view. Java has the same issue but hides it with the runtime `ArrayStoreException`. Scala chooses to make it a compile-time error instead, by defaulting to invariance.

So when *is* covariance safe? When the container only ever **produces** values of `A` — never consumes them. That's exactly the case for immutable read-only containers.

### Covariance — `+A`

Declare `class Box[+A]` and you tell the compiler: *`Box` is covariant in `A`. A `Box[Cat]` should be usable wherever a `Box[Animal]` is expected.* In exchange, the compiler enforces that `A` never appears in a *consumer* position — no method may take an `A` as a parameter.

In [ ]:
class Animal
class Cat extends Animal
class Dog extends Animal

// Read-only box: A appears only in output positions
class IBox[+A](val value: A)

val catBox: IBox[Cat]       = IBox(Cat())
val animalBox: IBox[Animal] = catBox   // ok — covariance allows widening

The standard-library `List`, `Vector`, `Option`, `Try`, and `Seq` are all covariant for this reason — they're immutable, so they only ever hand `A` values out, never accept new ones into existing instances. That's why `List[Cat]` is a `List[Animal]` for free.

### Contravariance — `-A`

Contravariance is the mirror image. Declare `class Sink[-A]` and you tell the compiler: *a `Sink[Animal]` should be usable wherever a `Sink[Cat]` is expected.* This *flips* the subtype relationship — the more general the parameter, the more places it fits.

It feels backwards until you think about it from the *user's* perspective. A consumer of `Cat`s can be satisfied by any consumer of `Animal`s — because a function that handles any animal certainly handles cats.

In [ ]:
trait Printer[-A]:
  def print(a: A): Unit

val animalPrinter: Printer[Animal] = new Printer[Animal]:
  def print(a: Animal): Unit = println(s"animal of class ${a.getClass.getSimpleName}")

// A Printer[Animal] can stand in wherever a Printer[Cat] is needed
val catPrinter: Printer[Cat] = animalPrinter
catPrinter.print(Cat())   // works — Cat is an Animal, the printer handles it

### Producers and consumers — a mental model

The shortcut for remembering which way is which:

- A type that **produces** `A` (read-only, hands `A` out) is naturally **covariant** in `A`. `Box[+A]`.
- A type that **consumes** `A` (write-only, takes `A` in) is naturally **contravariant** in `A`. `Sink[-A]`.
- A type that does both (mutable cell, read-write store) must stay **invariant**. `Box[A]`.

Java folks may know this as PECS — *Producer extends, Consumer super*. Scala bakes the same rule into the type system at the declaration site, instead of forcing it at every use site as Java's wildcards do.

### `Function1` — both at once

The clearest example of mixed variance in the standard library is the function type itself. `Function1[-T, +R]` is contravariant in its input and covariant in its output.

In [ ]:
class Animal
class Cat extends Animal

// A function that takes Animal and returns Cat
val af: Animal => Cat = (_: Animal) => Cat()

// It's usable wherever a Cat => Animal is expected:
//   input  — Cat => Animal expects Cat in,    af takes any Animal (wider)  ✓
//   output — Cat => Animal expects Animal out, af returns Cat (narrower)   ✓
val cf: Cat => Animal = af

Read that example slowly — it's the whole point of variance in two lines. A function that *takes more* and *gives more specific* is a perfectly valid substitute for one that takes less and gives less specific. The input is contravariant; the output is covariant. The same logic applies to every function arity in Scala — `Function2`, `Function3`, and so on.

## Variance positions — what the compiler actually checks

When you write `+A` or `-A`, the compiler enforces a strict rule: `A` may only appear in *positions* that are consistent with its declared variance.

- Method **return types** are *covariant positions*. A covariant `+A` may appear there.
- Method **parameter types** are *contravariant positions*. A contravariant `-A` may appear there.
- A `val` or `def` with no parameters returning `A` is a covariant position.
- A `var` is *both* — read and write — so it forces invariance.

Violate the rule and the compiler stops you. Try writing a method that takes `A` on a covariant class:

In [ ]:
// This is rejected by the compiler:
//
// class IBox[+A](val value: A):
//   def replace(a: A): IBox[A] = IBox(a)
//                  ^^^^^^^
//   covariant type A occurs in contravariant position in type A of value a

### The lower-bound escape hatch

The fix is to introduce a *fresh* type parameter `B` that's a supertype of `A`, and accept that. The method widens the box's element type rather than smuggling a narrower one in. This is exactly the pattern `List#::` uses to prepend an element to a covariant `List`.

In [ ]:
class IBox[+A](val value: A):
  // 'replace' returns a possibly-wider box. B >: A means B is a supertype of A.
  def replace[B >: A](b: B): IBox[B] = IBox(b)

class Animal
class Cat extends Animal
class Dog extends Animal

val catBox: IBox[Cat]    = IBox(Cat())
val mixed: IBox[Animal]  = catBox.replace(Dog())   // widened to Animal automatically

The pattern is small and worth memorising: **covariant class + lower-bounded method = safe write that widens the type.** You'll see it everywhere in the immutable collections.

## Where the standard library uses variance

A quick cheat sheet of common parameterised types and their variance:

```
  List[+A]            covariant   — immutable, read-only
  Vector[+A]          covariant   — immutable, read-only
  Seq[+A]             covariant   — immutable interface
  Option[+A]          covariant   — immutable
  Try[+A]             covariant   — immutable
  Either[+E, +A]      covariant in both — immutable
  Set[A]              invariant   — element type drives hashing/equality
  Map[K, +V]          invariant in K, covariant in V
  Array[A]            invariant   — mutable, JVM-backed
  Function1[-T, +R]   contravariant in input, covariant in output
```

Notice `Array` is invariant even though it *looks* like a list — because it's mutable. And `Set` is invariant because its element type participates in `hashCode`/`equals` lookups, where covariance would let you accidentally search the wrong bucket.

## Type erasure — the asterisk on everything above

Generics in Scala (like in Java) are a *compile-time* feature. At runtime, the JVM doesn't know that a `List` was a `List[Cat]` — the element type is *erased*. So variance, type bounds, and parameterised types only exist to keep your source code honest. At runtime, `List[Cat]` and `List[Dog]` are indistinguishable.

The practical consequence is one you already met in notebook 08 on pattern matching: a type pattern like `case xs: List[Int]` actually only checks `List` — the compiler will warn you. The general fix is to design so that static types carry the element-type information through your code, rather than relying on runtime reflection.

## Putting it together — a generic immutable stack

All the pieces in one small example: a covariant immutable stack with type-safe push and pop.

In [ ]:
sealed trait Stack[+A]:
  def push[B >: A](b: B): Stack[B] = NonEmpty(b, this)
  def pop: Option[(A, Stack[A])] = this match
    case Empty            => None
    case NonEmpty(h, t)   => Some((h, t))

case object Empty extends Stack[Nothing]
case class NonEmpty[+A](head: A, tail: Stack[A]) extends Stack[A]

val s1: Stack[Cat]     = Empty.push(Cat())
val s2: Stack[Animal]  = s1.push(Dog())     // widens to Animal via lower bound
s2.pop                                       // Some((Dog(), Stack[Animal](Cat())))

Three details to register from that example:

- `Stack[+A]` is covariant — `Stack[Cat]` is a `Stack[Animal]`.
- `push[B >: A]` uses a lower bound so adding a `Dog` to a `Stack[Cat]` widens the result to `Stack[Animal]` automatically.
- `Empty extends Stack[Nothing]` — `Nothing` is Scala's bottom type, a subtype of every type, so `Stack[Nothing]` is a subtype of `Stack[A]` for every `A`. That's how a single `Empty` singleton serves as the empty stack of every element type at once. This is the same reason `Nil: List[Nothing]` works for every `List[A]`.

## What's next

Notebook 11 covers **givens and extensions** — Scala 3's reworked take on implicit values and conversions. Generics let you abstract over types; givens let the compiler *provide* values for those types automatically, which is the foundation of type classes, contextual abstractions, and most of the ergonomic syntax in libraries like Cats and ZIO.